# Laboratorio 04 — CNN para ojos abiertos y cerrados (opcional)
**Curso:** Deep Learning y Agentes Inteligentes
**Docente:** Ing. Julián René Muñoz Burbano
**Proyecto:** DriveGuard Lab — Detección de somnolencia y riesgo del conductor

**Objetivo:** entrenar una red neuronal convolucional (CNN) real que clasifique recortes de ojos como abiertos o
cerrados, evaluarla y compararla con la regla del EAR. Este laboratorio completa la CNN *no entrenada* que se
mostró en el laboratorio 02 de SentimentLab V3.

**Datos (elija una opción):**
1. **MRL Eye Dataset** (Fusek, 2018): ~85 000 recortes de ojos en infrarrojo. Descárguelo desde
   http://mrl.cs.vsb.cz/eyedataset (o un espejo en Kaggle) y descomprímalo en `data/ojos/mrl/`. La etiqueta está
   en el nombre del archivo: `s0001_00001_0_0_1_0_0_01.png` → el 5.º campo es el estado del ojo (0 cerrado, 1 abierto).
2. **Capturas propias:** la sección 3 guarda recortes de sus ojos en `data/ojos/propios/abiertos` y `.../cerrados`.
3. **Modo demostración:** si no hay datos, se generan ojos sintéticos *solo para comprobar que el código funciona*.
   Sus métricas **no** representan el desempeño real.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Directorio del proyecto:', ROOT)
print('Python:', sys.executable)

In [ ]:
import random, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from core.config import OJOS_DIR, MODELS_DIR, CNN_OJOS_PATH
from core.ojos_cnn import crear_cnn, preparar, TAM, CLASES
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 1. Cargar el dataset

In [ ]:
def cargar_mrl(carpeta, max_por_clase=6000):
    rutas = {0: [], 1: []}
    for p in carpeta.rglob('*.png'):
        partes = p.stem.split('_')
        if len(partes) >= 5 and partes[4] in ('0', '1'):
            rutas[int(partes[4])].append(p)
    salida = []
    for clase, lista in rutas.items():
        random.shuffle(lista)
        salida += [(p, clase) for p in lista[:max_por_clase]]
    return salida

def cargar_propios(carpeta):
    salida = []
    for clase, nombre in enumerate(CLASES):     # 0 cerrado, 1 abierto
        salida += [(p, clase) for p in (carpeta / f'{nombre}s').glob('*.png')]
    return salida

def ojos_sinteticos(n=1500, semilla=0):
    rng = np.random.default_rng(semilla); X, y = [], []
    for i in range(n):
        img = np.full((TAM, TAM), rng.integers(90, 170), np.uint8)
        img = np.clip(img + rng.normal(0, 12, img.shape), 0, 255).astype(np.uint8)
        cx, cy = TAM // 2 + rng.integers(-5, 6), TAM // 2 + rng.integers(-4, 5)
        if i % 2:   # abierto
            cv2.ellipse(img, (cx, cy), (int(rng.integers(18, 26)), int(rng.integers(8, 13))), 0, 0, 360, int(rng.integers(190, 240)), -1)
            cv2.circle(img, (cx + int(rng.integers(-4, 5)), cy), int(rng.integers(5, 8)), int(rng.integers(10, 60)), -1)
        else:       # cerrado
            cv2.ellipse(img, (cx, cy), (int(rng.integers(18, 26)), int(rng.integers(2, 5))), 0, 0, 180, int(rng.integers(20, 70)), 2)
        X.append(cv2.GaussianBlur(img, (3, 3), 0)); y.append(i % 2)
    return X, y

def leer(ruta):
    img = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (TAM, TAM), interpolation=cv2.INTER_AREA)

mrl, propios = cargar_mrl(OJOS_DIR / 'mrl'), cargar_propios(OJOS_DIR / 'propios')
if mrl or propios:
    MODO_DEMO = False
    muestras = mrl + propios
    X = [leer(p) for p, _ in muestras]; y = [c for _, c in muestras]
    origen = 'MRL' if mrl else 'capturas propias'
else:
    MODO_DEMO = True
    X, y = ojos_sinteticos()
    origen = 'SINTÉTICO (solo demostración)'
X, y = np.array(X), np.array(y)
print(f'Origen: {origen} | imágenes: {len(X)} | cerrados: {(y==0).sum()} | abiertos: {(y==1).sum()}')

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for fila, clase in enumerate([0, 1]):
    idx = np.random.choice(np.where(y == clase)[0], 8, replace=False)
    for ax, i in zip(axes[fila], idx):
        ax.imshow(X[i], cmap='gray'); ax.axis('off')
    axes[fila][0].set_title(CLASES[clase], loc='left')
plt.tight_layout(); plt.show()

## 2. Capturar recortes de sus propios ojos (opcional)
Con `USE_CAMERA = True` se guardan recortes usando MediaPipe: primero con ojos abiertos y luego cerrados.

In [ ]:
USE_CAMERA = False
SEGUNDOS_POR_CLASE = 15
if USE_CAMERA:
    from core.landmarks import DetectorRostro, recorte_ojo, OJO_DERECHO, OJO_IZQUIERDO
    cap = cv2.VideoCapture(0)
    with DetectorRostro(modo='video') as det:
        for clase in ['abierto', 'cerrado']:
            destino = OJOS_DIR / 'propios' / f'{clase}s'; destino.mkdir(parents=True, exist_ok=True)
            input(f'Pulse Enter y mantenga los ojos {clase.upper()}S durante {SEGUNDOS_POR_CLASE} s...')
            t0, n = time.time(), 0
            while time.time() - t0 < SEGUNDOS_POR_CLASE:
                ok, frame = cap.read()
                if not ok: break
                r = det.procesar(frame)
                if r is None: continue
                for idx in (OJO_DERECHO, OJO_IZQUIERDO):
                    rec = recorte_ojo(frame, r.puntos, idx)
                    if rec is not None:
                        cv2.imwrite(str(destino / f'{clase}_{int(time.time()*1000)}_{n}.png'), rec); n += 1
            print(f'{n} recortes de ojos {clase}s guardados en {destino}')
    cap.release()
    print('Vuelva a ejecutar la celda de carga del dataset.')
else:
    print('Captura desactivada (USE_CAMERA = False)')

## 3. División entrenamiento / validación / prueba
Se estratifica por clase. Con el MRL, lo ideal es dividir **por sujeto** para no evaluar con personas vistas en el entrenamiento (actividad 2).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=0)

class Ojos(Dataset):
    def __init__(self, X, y, aumentar=False):
        self.X, self.y, self.aumentar = X, y, aumentar
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        img = self.X[i]
        if self.aumentar:
            if random.random() < .5: img = img[:, ::-1]
            img = np.clip(img.astype(np.float32) * random.uniform(.7, 1.3) + random.uniform(-20, 20), 0, 255).astype(np.uint8)
        return preparar([np.ascontiguousarray(img)])[0], int(self.y[i])

dl_tr = DataLoader(Ojos(X_tr, y_tr, True), batch_size=64, shuffle=True)
dl_va = DataLoader(Ojos(X_va, y_va), batch_size=256)
dl_te = DataLoader(Ojos(X_te, y_te), batch_size=256)
print(len(X_tr), 'entrenamiento |', len(X_va), 'validación |', len(X_te), 'prueba')

## 4. Arquitectura de la CNN
Tres bloques *convolución → normalización → ReLU → max-pooling*, *global average pooling* y una capa lineal.

In [ ]:
modelo = crear_cnn()
print(modelo)
print('Parámetros entrenables:', sum(p.numel() for p in modelo.parameters() if p.requires_grad))

## 5. Entrenamiento

In [ ]:
EPOCAS = 4 if MODO_DEMO else 10
opt = torch.optim.Adam(modelo.parameters(), lr=1e-3)
perdida_fn = nn.CrossEntropyLoss()

def evaluar(dl):
    modelo.eval(); pred, real = [], []
    with torch.no_grad():
        for xb, yb in dl:
            pred += modelo(xb).argmax(1).tolist(); real += yb.tolist()
    return np.array(real), np.array(pred)

historial = []
for epoca in range(1, EPOCAS + 1):
    modelo.train(); total = 0
    for xb, yb in dl_tr:
        opt.zero_grad(); loss = perdida_fn(modelo(xb), yb); loss.backward(); opt.step()
        total += loss.item() * len(xb)
    real, pred = evaluar(dl_va)
    historial.append({'epoca': epoca, 'perdida': total / len(dl_tr.dataset), 'acc_val': (real == pred).mean()})
    print(f"Época {epoca:2d} | pérdida {historial[-1]['perdida']:.4f} | exactitud validación {historial[-1]['acc_val']:.3f}")
h = pd.DataFrame(historial)
h.plot(x='epoca', y=['perdida', 'acc_val'], secondary_y='acc_val', figsize=(8, 3), marker='o'); plt.show()

## 6. Evaluación en el conjunto de prueba

In [ ]:
real, pred = evaluar(dl_te)
print(classification_report(real, pred, target_names=CLASES, digits=3))
ConfusionMatrixDisplay(confusion_matrix(real, pred), display_labels=CLASES).plot(cmap='Blues'); plt.show()
if MODO_DEMO:
    print('AVISO: datos sintéticos. Estas métricas solo prueban que el código funciona.')

## 7. Guardar el modelo y medir velocidad

In [ ]:
torch.save({'modelo': modelo.state_dict(), 'origen': origen, 'tam': TAM, 'clases': CLASES,
            'f1_prueba': float(f1_score(real, pred, average='macro'))}, CNN_OJOS_PATH)
print('Modelo guardado en', CNN_OJOS_PATH)

from core.ojos_cnn import ClasificadorOjos
clf = ClasificadorOjos(CNN_OJOS_PATH)
lote = [X_te[0], X_te[1]]
clf.prob_cerrado(lote)
t0 = time.perf_counter()
for _ in range(50): clf.prob_cerrado(lote)
print(f'Inferencia de 2 ojos: {(time.perf_counter() - t0) / 50 * 1000:.1f} ms por fotograma')
if MODO_DEMO:
    print('No active OJOS_METODO=cnn con este modelo de demostración: entrénelo con datos reales.')

## 8. Comparación EAR vs. CNN en su propio video
Grabe un video con segmentos **etiquetados por tiempo** (por ejemplo, 0–10 s ojos abiertos, 10–20 s cerrados,
20–30 s abiertos...). Defina los intervalos de ojos cerrados en `CERRADOS` y ejecute la celda. Para cada fotograma se
calcula la decisión del EAR (umbral = 75 % del EAR base) y la de la CNN, y se comparan con la etiqueta.

In [ ]:
from core.config import VIDEOS_DIR
VIDEO = VIDEOS_DIR / 'ojos_etiquetado.mp4'
CERRADOS = [(10, 20), (30, 35)]      # segundos con ojos cerrados (ajuste a su grabación)
if VIDEO.exists() and CNN_OJOS_PATH.exists():
    from core.landmarks import DetectorRostro, recorte_ojo, OJO_DERECHO, OJO_IZQUIERDO
    cap = cv2.VideoCapture(str(VIDEO)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    filas, i = [], 0
    with DetectorRostro(modo='video') as det:
        while True:
            ok, frame = cap.read()
            if not ok: break
            t = i / fps; i += 1
            r = det.procesar(frame, int(t * 1000))
            if r is None: continue
            rec = [x for x in (recorte_ojo(frame, r.puntos, OJO_DERECHO), recorte_ojo(frame, r.puntos, OJO_IZQUIERDO)) if x is not None]
            filas.append({'t': t, 'ear': r.ear, 'p_cnn': clf.prob_cerrado(rec).mean() if rec else np.nan,
                          'real': any(a <= t < b for a, b in CERRADOS)})
    cap.release()
    comp = pd.DataFrame(filas).dropna()
    base = comp[comp.t < 5].ear.median()
    comp['ear_cerrado'] = comp.ear < 0.75 * base
    comp['cnn_cerrado'] = comp.p_cnn > 0.5
    for metodo in ['ear_cerrado', 'cnn_cerrado']:
        print(metodo, '| F1 =', round(f1_score(comp.real, comp[metodo]), 3),
              '| exactitud =', round((comp.real == comp[metodo]).mean(), 3))
else:
    print(f'Para comparar grabe {VIDEO} y entrene la CNN con datos reales.')

## 9. Usar la CNN en la aplicación
En `.env`: `OJOS_METODO=cnn`. Al iniciar la cámara, `ProcesadorConductor` usará la CNN para decidir si los ojos
están cerrados (el EAR se sigue mostrando como referencia). Si el modelo no existe, la app vuelve al EAR y lo informa.

## 10. Actividades
1. Entrene con el MRL Eye Dataset completo y reporte exactitud, F1 y matriz de confusión.
2. Divida el MRL **por sujeto** (primer campo del nombre) y compare con la división aleatoria. ¿Por qué cambia?
3. Evalúe el modelo entrenado con MRL (infrarrojo) sobre sus capturas a color. ¿Generaliza?
4. Ajuste fino: entrene con MRL y luego unas épocas con sus capturas.
5. Compare EAR vs. CNN en exactitud, robustez con gafas y tiempo de inferencia. ¿Cuál usaría en un vehículo y por qué?